# Model Development

- Model training and evaluation
- Hyperparameter tuning
- Model comparison


In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
from pathlib import Path

current_dir = Path().resolve()
if current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

sys.path.insert(0, str(project_root))
from src.data.data_loader import DataLoader


In [2]:
data_path = None
possible_paths = [
    'data/processed/train.csv',
    '../data/processed/train.csv',
    '../../data/processed/train.csv'
]

for path in possible_paths:
    if os.path.exists(path):
        data_path = path
        break

if data_path is None:
    raise FileNotFoundError(f"Could not find processed data. Tried: {possible_paths}")

print(f"Loading data from: {data_path}")

train = pd.read_csv(data_path.replace('train.csv', 'train.csv'))
val = pd.read_csv(data_path.replace('train.csv', 'validation.csv'))
test = pd.read_csv(data_path.replace('train.csv', 'test.csv'))

print(f"Train shape: {train.shape}")
print(f"Validation shape: {val.shape}")
print(f"Test shape: {test.shape}")

# Prepare features and target
X_train = train.drop('Class', axis=1)
y_train = train['Class']
X_val = val.drop('Class', axis=1)
y_val = val['Class']
X_test = test.drop('Class', axis=1)
y_test = test['Class']

print(f"\nFraud rate - Train: {y_train.mean():.4f}, Val: {y_val.mean():.4f}, Test: {y_test.mean():.4f}")


Loading data from: ../data/processed/train.csv
Train shape: (199364, 31)
Validation shape: (28481, 31)
Test shape: (56962, 31)

Fraud rate - Train: 0.0017, Val: 0.0017, Test: 0.0017


In [ ]:
# Train Logistic Regression
print("Training Logistic Regression...")
lr_model = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr_model.fit(X_train, y_train)

# Predictions
lr_pred = lr_model.predict(X_test)
lr_pred_proba = lr_model.predict_proba(X_test)[:, 1]

# Evaluation
print("\nLogistic Regression Results:")
print(classification_report(y_test, lr_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, lr_pred_proba):.4f}")


In [ ]:
# Train Random Forest
print("Training Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

# Predictions
rf_pred = rf_model.predict(X_test)
rf_pred_proba = rf_model.predict_proba(X_test)[:, 1]

# Evaluation
print("\nRandom Forest Results:")
print(classification_report(y_test, rf_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, rf_pred_proba):.4f}")


In [ ]:
# Model Comparison
models = {
    'Logistic Regression': (lr_pred, lr_pred_proba),
    'Random Forest': (rf_pred, rf_pred_proba)
}

print("=" * 60)
print("MODEL COMPARISON")
print("=" * 60)

results = []
for name, (pred, pred_proba) in models.items():
    roc_auc = roc_auc_score(y_test, pred_proba)
    report = classification_report(y_test, pred, output_dict=True)
    results.append({
        'Model': name,
        'ROC-AUC': roc_auc,
        'Precision': report['1']['precision'],
        'Recall': report['1']['recall'],
        'F1-Score': report['1']['f1-score']
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))


In [ ]:
# Plot ROC Curves
plt.figure(figsize=(10, 6))

for name, (_, pred_proba) in models.items():
    fpr, tpr, _ = roc_curve(y_test, pred_proba)
    auc = roc_auc_score(y_test, pred_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Model Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for idx, (name, (pred, _)) in enumerate(models.items()):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx])
    axes[idx].set_title(f'{name} - Confusion Matrix')
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()
